# Housing Price Prediction

This notebook loads the housing dataset, trains several regression models, and compares their predictive performance.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

try:
    import xgboost as xgb
    XGBOOST_AVAILABLE = True
except Exception as e:
    xgb = None
    XGBOOST_AVAILABLE = False
    print(f"XGBoost could not be imported: {e}")

# Load the dataset
df = pd.read_csv('housing.csv')
df.head()

XGBoost could not be imported: 
XGBoost Library (libxgboost.dylib) could not be loaded.
Likely causes:
  * OpenMP runtime is not installed
    - vcomp140.dll or libgomp-1.dll for Windows
    - libomp.dylib for Mac OSX
    - libgomp.so for Linux and other UNIX-like OSes
    Mac OSX users: Run `brew install libomp` to install OpenMP runtime.

  * You are running 32-bit Python on a 64-bit OS

Error message(s): ["dlopen(/Users/mac/Downloads/house-price-prediction/.venv/lib/python3.9/site-packages/xgboost/lib/libxgboost.dylib, 0x0006): Library not loaded: @rpath/libomp.dylib\n  Referenced from: <89AD948E-E564-3266-867D-7AF89D6488F0> /Users/mac/Downloads/house-price-prediction/.venv/lib/python3.9/site-packages/xgboost/lib/libxgboost.dylib\n  Reason: tried: '/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/System/Volumes/Preboo

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


In [ ]:
# Split features and target
X = df.drop(columns=['median_house_value'])
y = df['median_house_value']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


def build_model(regressor):
    numeric_features = X.select_dtypes(exclude=['object']).columns
    categorical_features = ['ocean_proximity']

    numeric_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])

    categorical_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ])

    preprocessor = ColumnTransformer(transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

    return Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('regressor', regressor)
    ])


models = {
    'Linear Regression': build_model(LinearRegression()),
    'Decision Tree': build_model(DecisionTreeRegressor(random_state=42)),
    'Random Forest': build_model(RandomForestRegressor(n_estimators=200, random_state=42)),
}

if XGBOOST_AVAILABLE:
    models['XGBoost'] = build_model(xgb.XGBRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=5,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1
    ))

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    rmse = mean_squared_error(y_test, predictions) ** 0.5
    r2 = r2_score(y_test, predictions)
    results[name] = {'rmse': rmse, 'r2': r2}

    print(f'{name}')
    print(f'RMSE: {rmse:.2f}')
    print(f'R^2 Score: {r2:.3f}')
    print('-' * 40)

best_model_name = min(results, key=lambda name: results[name]['rmse'])
best_model = models[best_model_name]
print(f'Best model by RMSE: {best_model_name}')

Linear Regression
RMSE: 70059.19
R^2 Score: 0.625
----------------------------------------
Random Forest
RMSE: 48781.91
R^2 Score: 0.818
----------------------------------------
Best model by RMSE: Random Forest


In [6]:
# Example prediction for a new sample
sample = pd.DataFrame([
    {
        'longitude': -122.23,
        'latitude': 37.88,
        'housing_median_age': 41,
        'total_rooms': 880,
        'total_bedrooms': 129,
        'population': 322,
        'households': 126,
        'median_income': 8.3252,
        'ocean_proximity': 'NEAR BAY'
    }
])

predicted_value = best_model.predict(sample)[0]
print(f'Predicted house value: ${predicted_value:,.2f}')

Predicted house value: $431,528.85
